## Step 7: Save Processed Data

In [7]:
# Data Preprocessing - AgriPrice AI

# This notebook demonstrates the complete data preprocessing pipeline for the AgriPrice AI project.

## Steps:
# 1. **Load Raw Data** - Import CSV from government sources
# 2. **Clean Data** - Handle missing values, remove outliers, sort by date
# 3. **Feature Engineering** - Add lag, rolling averages, seasonal indicators
# 4. **Save Processed Data** - Output cleaned & enriched dataset for modeling

In [1]:
import pandas as pd
import numpy as np
import os

# Define paths relative to notebook location
notebook_dir = os.getcwd()
base_dir = os.path.dirname(notebook_dir)  # Go up from notebook/ to project root
raw_data_path = os.path.join(base_dir, "data", "raw_data.csv")
processed_data_path = os.path.join(base_dir, "data", "processed_data.csv")

print(f"Notebook dir: {notebook_dir}")
print(f"Base dir: {base_dir}")
print(f"Loading data from: {raw_data_path}")
data = pd.read_csv(raw_data_path)
print(f"Raw data shape: {data.shape}")
print(f"\nFirst 5 rows:")
print(data.head())

Notebook dir: c:\Users\jabhi\Downloads\main-main\main-main\notebook
Base dir: c:\Users\jabhi\Downloads\main-main\main-main
Loading data from: c:\Users\jabhi\Downloads\main-main\main-main\data\raw_data.csv
Raw data shape: (7, 4)

First 5 rows:
         Date Commodity  Price  Volume
0  2022-01-01     Onion     50    2000
1  2022-01-02     Onion     52    2100
2  2022-01-03     Onion     51    2150
3  2022-01-04     Onion     49    1900
4  2022-01-05     Onion     48    1800


In [2]:
# Check data types and missing values
print("Data types:")
print(data.dtypes)
print("\nMissing values:")
print(data.isnull().sum())
print("\nBasic statistics:")
print(data.describe())

Data types:
Date           str
Commodity      str
Price        int64
Volume       int64
dtype: object

Missing values:
Date         0
Commodity    0
Price        0
Volume       0
dtype: int64

Basic statistics:
           Price       Volume
count   7.000000     7.000000
mean   50.142857  2014.285714
std     1.345185   124.880896
min    48.000000  1800.000000
25%    49.500000  1950.000000
50%    50.000000  2050.000000
75%    51.000000  2100.000000
max    52.000000  2150.000000


In [3]:
# Step 1: Normalize Price column (handle Modal_Price or Price)
if "Modal_Price" in data.columns:
    data["Price"] = pd.to_numeric(data["Modal_Price"], errors="coerce")
elif "Price" in data.columns:
    data["Price"] = pd.to_numeric(data["Price"], errors="coerce")
else:
    raise ValueError("Data must contain either 'Price' or 'Modal_Price' column")

# Step 2: Convert Date to datetime
data["Date"] = pd.to_datetime(data["Date"], errors="coerce")

# Step 3: Drop rows with missing critical values
data = data.dropna(subset=["Date", "Price"])
print(f"After removing NaN: {len(data)} rows")

# Step 4: Sort by Date
data = data.sort_values(by="Date").reset_index(drop=True)

# Step 5: Forward fill missing values
data = data.ffill()

print("Data after cleaning:")
print(data.head())

After removing NaN: 7 rows
Data after cleaning:
        Date Commodity  Price  Volume
0 2022-01-01     Onion     50    2000
1 2022-01-02     Onion     52    2100
2 2022-01-03     Onion     51    2150
3 2022-01-04     Onion     49    1900
4 2022-01-05     Onion     48    1800


In [4]:
# Step 6: Remove outliers using IQR method
q1 = data["Price"].quantile(0.25)
q3 = data["Price"].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

print(f"IQR bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")
print(f"Rows before outlier removal: {len(data)}")

data = data[(data["Price"] >= lower_bound) & (data["Price"] <= upper_bound)]
print(f"Rows after outlier removal: {len(data)}")
print("\nPrice statistics after cleaning:")
print(data["Price"].describe())

IQR bounds: [47.25, 53.25]
Rows before outlier removal: 7
Rows after outlier removal: 7

Price statistics after cleaning:
count     7.000000
mean     50.142857
std       1.345185
min      48.000000
25%      49.500000
50%      50.000000
75%      51.000000
max      52.000000
Name: Price, dtype: float64


In [5]:
# Step 7: Feature Engineering
print("Adding engineered features...")

# Lag features (previous values)
data["price_lag_7"] = data["Price"].shift(7)
data["price_lag_14"] = data["Price"].shift(14)
data["price_lag_30"] = data["Price"].shift(30)

# Rolling statistics (smoothed trends)
data["rolling_mean_7"] = data["Price"].rolling(window=7, min_periods=2).mean()
data["rolling_mean_30"] = data["Price"].rolling(window=30, min_periods=2).mean()
data["rolling_std_7"] = data["Price"].rolling(window=7, min_periods=2).std()

# Date-based features
data["month"] = data["Date"].dt.month
data["quarter"] = data["Date"].dt.quarter
data["year"] = data["Date"].dt.year

# Seasonal indicators
data["is_harvest_season"] = data["month"].isin([11, 12, 1]).astype(int)
data["is_festival_season"] = data["month"].isin([3, 4, 10]).astype(int)

# Forward/backward fill any remaining NaN in features
feature_cols = [
    "price_lag_7", "price_lag_14", "price_lag_30",
    "rolling_mean_7", "rolling_mean_30", "rolling_std_7"
]
for col in feature_cols:
    data[col] = data[col].bfill().ffill()

print(f"\nFeatures added. Shape: {data.shape}")
print(f"\nNew columns: {[col for col in data.columns if col not in ['Date', 'Price', 'Commodity', 'Volume']]}")
print("\nFinal processed data (first 5 rows):")
print(data.head())

Adding engineered features...

Features added. Shape: (7, 15)

New columns: ['price_lag_7', 'price_lag_14', 'price_lag_30', 'rolling_mean_7', 'rolling_mean_30', 'rolling_std_7', 'month', 'quarter', 'year', 'is_harvest_season', 'is_festival_season']

Final processed data (first 5 rows):
        Date Commodity  Price  Volume  price_lag_7  price_lag_14  \
0 2022-01-01     Onion     50    2000          NaN           NaN   
1 2022-01-02     Onion     52    2100          NaN           NaN   
2 2022-01-03     Onion     51    2150          NaN           NaN   
3 2022-01-04     Onion     49    1900          NaN           NaN   
4 2022-01-05     Onion     48    1800          NaN           NaN   

   price_lag_30  rolling_mean_7  rolling_mean_30  rolling_std_7  month  \
0           NaN            51.0             51.0       1.414214      1   
1           NaN            51.0             51.0       1.414214      1   
2           NaN            51.0             51.0       1.000000      1   
3       

In [6]:
# Save processed data
os.makedirs(os.path.dirname(processed_data_path), exist_ok=True)
data.to_csv(processed_data_path, index=False)
print(f"✅ Processed data saved to: {processed_data_path}")
print(f"Total rows: {len(data)}")
print(f"Total columns: {len(data.columns)}")
print(f"\nColumns: {list(data.columns)}")

✅ Processed data saved to: c:\Users\jabhi\Downloads\main-main\main-main\data\processed_data.csv
Total rows: 7
Total columns: 15

Columns: ['Date', 'Commodity', 'Price', 'Volume', 'price_lag_7', 'price_lag_14', 'price_lag_30', 'rolling_mean_7', 'rolling_mean_30', 'rolling_std_7', 'month', 'quarter', 'year', 'is_harvest_season', 'is_festival_season']
